# ConfRover training pipeline — component checks (PACE)

Validates the **new training code** on the bundled ATLAS test data. Runs offline
(no download, no MSA server), so it works on a PACE compute node with no internet.

**Before you start (PACE Phoenix, interactive):**
- Grab a GPU: `salloc` (see `scripts/phoenix_interactive.sh`) or Open OnDemand → Jupyter (1 GPU, 8 cores, 32 GB, QOS=embers).
- Activate the conda env from the top-level README (`.[openfold]` installed).
- Launch Jupyter from the repo root.

This notebook checks: (1) unit tests, (2) backbone-atom loss gating, (3) dataset
upgrades, (4) a GPU forward pass with the new loss, (5) checkpoint round-trip into
a plain `ConfRover`, (6) the eval metrics. It also writes a checkpoint that
`pace_02_train_and_eval.ipynb` can reuse.

In [ ]:
import os, sys
from pathlib import Path

# Resolve repo root whether the notebook runs from examples/ or repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo root:", REPO_ROOT)

# Bundled test data -- everything below runs offline (no ATLAS download / MSA).
TESTDATA   = REPO_ROOT / "tests" / "test_data"
ATLAS_ROOT = TESTDATA / "atlas"            # <case>/<case>.pdb + <case>_prod_R1_fit.xtc
REPR_ROOT  = TESTDATA / "openfold_repr"    # cached OpenFold features (seqres-keyed)

# The four bundled proteins that have BOTH an ATLAS xtc and cached OpenFold repr.
BUNDLED = {
    "7jfl_C": "SALQDLLRTLKSPSSPQQQQQVLNILKSNPQLMAAFIKQRTAKYVAN",
    "6okd_C": "GSREGCASRCMKYNDELEKCEARMMSMSNTEEDCEQELEDLLYCLDHCHSQ",
    "6ro6_A": "MEGALPKGLSDLIADPTLGPQITPDWVRTLSRIELRGKRPRDKQDWYEIYLHLKRILS",
    "7lp1_A": "VTQSFLPPGWEMRIAPNGRPFFIDHNTKTTTWEDPRLKF",
}

CACHE = Path(os.environ.get("CONFROVER_CACHE_DIR", Path.home() / "scratch" / "confrover_cache")).expanduser()
CACHE.mkdir(parents=True, exist_ok=True)

# generate() unconditionally runs install_cutlass(), which git-clones CUTLASS if
# it's missing -- that fails on an offline compute node. Evo-attention kernels are
# OFF in these configs, so CUTLASS is never actually used; point CUTLASS_PATH at an
# existing (stub) dir so the clone is skipped.
os.environ.setdefault("CUTLASS_PATH", str(CACHE / "cutlass_stub"))
Path(os.environ["CUTLASS_PATH"]).mkdir(parents=True, exist_ok=True)

import torch
torch.set_float32_matmul_precision("high")
assert torch.cuda.is_available(), "No GPU visible -- request an interactive GPU (salloc) first."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Unit tests
Fast CPU/GPU tests for the new code (loss, dataset logic, metrics).

In [ ]:
# The `slow` marker (full-model checkpoint round-trip) is deselected here; we do
# a live round-trip in section 5 instead.
!pytest tests/train -m "not slow" -q

## 2. Backbone-atom loss is gated on the diffusion time
`loss_bb_atom` must be 0 when every frame is noisy (`t > t_bb_threshold`) and > 0 when frames are near-clean (`t < t_bb_threshold`).

In [ ]:
from confrover.train.loss import SE3DiffusionLoss, LossWeights

def synthetic_inputs(t_value, BF=2, L=6, seed=0):
    g = torch.Generator().manual_seed(seed)
    r = lambda *s: torch.randn(*s, generator=g)
    return dict(
        t=torch.full((BF,), float(t_value)),
        rigids_mask=torch.ones(BF, L),
        torsion_angles_mask=torch.ones(BF, L, 7),
        pred_rigids_0=None,
        pred_torsion_sin_cos=r(BF, L, 7, 2),
        pred_atom14=r(BF, L, 14, 3).requires_grad_(True),
        pred_rot_score=r(BF, L, 3).requires_grad_(True),
        pred_trans_score=r(BF, L, 3).requires_grad_(True),
        pred_sidechain_frame=r(BF, L, 8, 4, 4),
        gt_feat=dict(
            gt_rot_score=r(BF, L, 3), gt_trans_score=r(BF, L, 3),
            rot_score_scaling=torch.ones(BF), trans_score_scaling=torch.ones(BF),
            atom14_gt_positions=r(BF, L, 14, 3),
        ),
    )

loss_mod = SE3DiffusionLoss(LossWeights(rot=1.0, trans=1.0, bb_atom=0.25))
_, aux_hi = loss_mod(**synthetic_inputs(t_value=0.9))   # all noisy -> bb gated off
loss_lo, aux_lo = loss_mod(**synthetic_inputs(t_value=0.05))  # near-clean -> bb active
print("t=0.9  loss_bb_atom =", float(aux_hi.get("loss_bb_atom", 0.0)))
print("t=0.05 loss_bb_atom =", float(aux_lo["loss_bb_atom"]))
assert float(aux_hi.get("loss_bb_atom", 0.0)) == 0.0
assert float(aux_lo["loss_bb_atom"]) > 0.0
loss_lo.backward()  # differentiable through pred_atom14
print("OK: bb_atom gated correctly and differentiable")

## 3. Dataset: real frame count, multi-replicate, random stride

In [ ]:
from confrover.train.dataset import (
    _count_xtc_frames, TrajCaseConfig, TrajDatasetConfig, TrajDataset,
)

# Real trajectory length via mdtraj (was hardcoded before).
xtc = ATLAS_ROOT / "7jfl_C" / "7jfl_C_prod_R1_fit.xtc"
print("7jfl_C frames:", _count_xtc_frames(str(xtc)))

# Build a 4-protein dataset with random-stride + (single, here) replicate lists.
cases = [
    TrajCaseConfig(
        case_id=cid, seqres=seq,
        pdb_fpath=f"{cid}/{cid}.pdb",
        xtc_fpaths=[f"{cid}/{cid}_prod_R1_fit.xtc"],  # add R2/R3 when you have them
    )
    for cid, seq in BUNDLED.items()
]
cfg = TrajDatasetConfig(
    name="bundled4", n_frames=4, stride_in_10ps=30,
    strides_in_10ps=[10, 20, 30], samples_per_epoch=8, cases=cases,
)
from confrover.data.pretrain_repr.openfold.loader import OpenFoldReprLoader
repr_loader = OpenFoldReprLoader(repr_root=str(REPR_ROOT), num_recycles=3,
                                 load_single=True, load_pair=True, v1=False)
ds = TrajDataset(config=cfg, repr_loader=repr_loader, relpath_to=str(ATLAS_ROOT),
                 deterministic=False, batch_size=1, num_workers=0, shuffle=True)

n = _count_xtc_frames(str(xtc))
print("chosen strides (sampled):", sorted({ds._choose_stride(n) for _ in range(20)}))
print("replicate picks:", {ds._pick_xtc_path(cases[0]) for _ in range(10)})

item = ds[0]
for k, v in item.items():
    if hasattr(v, "shape"):
        print(f"  {k:26s} {tuple(v.shape)}")

## 4. GPU forward pass with the new loss
Instantiate `ConfRoverTrainable` from the training config and run one `_shared_step`. The aux dict now includes `loss_bb_atom`.

In [ ]:
import hydra
from omegaconf import OmegaConf

model_cfg = OmegaConf.load(REPO_ROOT / "src" / "confrover" / "configs" / "model" / "confrover_train.yaml")
model = hydra.utils.instantiate(model_cfg).to("cuda")
model.set_model_cfg(OmegaConf.to_container(model_cfg, resolve=True))
print("Trainable params: %.2fM" % (sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6))

from lightning.pytorch.utilities import move_data_to_device
# Deterministic single-protein window for a stable check.
ds_det = TrajDataset(config=cfg, repr_loader=repr_loader, relpath_to=str(ATLAS_ROOT),
                     deterministic=True, batch_size=1, num_workers=0, shuffle=False)
batch = move_data_to_device(TrajDataset.collate([ds_det[0]]), "cuda")
loss, aux = model._shared_step(batch, stage="train")
print("loss =", float(loss))
for k, v in aux.items():
    print(f"  {k:16s} = {float(v):.4f}")

## 5. Checkpoint round-trips into a plain `ConfRover`
`on_save_checkpoint` embeds an inference-shaped `model_cfg`; a plain `ConfRover` built from it must load the trained weights with `strict=True`.

In [ ]:
from confrover.model.confrover import ConfRover

ckpt = {"state_dict": model.state_dict()}
model.on_save_checkpoint(ckpt)               # injects model_cfg
assert "model_cfg" in ckpt
assert ckpt["model_cfg"]["_target_"].endswith("ConfRover")
assert ckpt["model_cfg"]["decoder"]["loss"] is None

inf_model = ConfRover.from_config(ckpt["model_cfg"])
missing, unexpected = inf_model.load_state_dict(ckpt["state_dict"], strict=False)
assert not missing and not unexpected, (missing, unexpected)
print("OK: state_dict loads strictly into a plain ConfRover")

# Save a .pt for the inference CLI / pace_02 notebook.
ckpt_path = CACHE / "component_check.pt"
torch.save(ckpt, ckpt_path)
print("Saved:", ckpt_path)

## 6. Eval metrics sanity
`compare_ensembles` scores an ensemble against itself as perfect, and degrades under noise.

In [ ]:
import numpy as np
from confrover.train.eval.metrics import compare_ensembles

rng = np.random.default_rng(0)
ens = rng.normal(size=(12, 25, 3)) * 4.0
same = compare_ensembles(ens, ens, max_pairwise=12)
noisy = compare_ensembles(ens + rng.normal(size=ens.shape) * 2.0, ens, max_pairwise=12)
print("identical : rmsf_r=%.3f  min_rmsd=%.3f  best_tm=%.3f" % (
    same["rmsf_pearson"], same["coverage_mean_min_rmsd"], same["coverage_mean_best_tmscore"]))
print("noised    : rmsf_r=%.3f  min_rmsd=%.3f  best_tm=%.3f" % (
    noisy["rmsf_pearson"], noisy["coverage_mean_min_rmsd"], noisy["coverage_mean_best_tmscore"]))
assert same["coverage_mean_min_rmsd"] < 1e-6 < noisy["coverage_mean_min_rmsd"]
print("OK: metrics behave")

✅ All component checks passed. Next: `pace_02_train_and_eval.ipynb` runs a short real training + generation + evaluation on the same bundled proteins.